# Dissecting the best flight duration model

You just set up a `CrossValidator` to find good parameters for the linear regression model predicting flight duration.

The model pipeline has multiple stages (objects of type `StringIndexer`, `OneHotEncoder`, `VectorAssembler` and `LinearRegression`), which operate in sequence. The stages are available as the `stages` attribute on the `pipeline` object. They are represented by a list and the stages are executed in the sequence in which they appear in the list.

Now you're going to take a closer look at the pipeline, split out the stages and use it to make predictions on the testing data.

The following objects have already been created:

- `cv` — a trained `CrossValidatorModel` object and
- `evaluator` — a `RegressionEvaluator` object.
  
The `flights` data have been randomly split into `flights_train` and `flights_test`.

## Instructions

- Retrieve the best model.
- Look at the stages in the best model.
- Isolate the linear regression stage and extract its parameters.
- Use the best model to generate predictions on the testing data and calculate the RMSE.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [4]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M4-EnsemblesAndPipelines/3_GridSearch/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0)).drop('mile')
flights = flights.sample(0.25, seed=13)
flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=13)

# from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoderEstimator, VectorAssembler
indexer = StringIndexer(inputCol='org', outputCol='org_idx')
# onehot = OneHotEncoder(inputCols=['org_idx'], outputCols=['org_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['org_idx'], outputCols=['org_dummy'])
assembler = VectorAssembler(inputCols=['km', 'org_dummy'], outputCol='features')

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline, PipelineModel

regression = LinearRegression(labelCol='duration')
evaluator = RegressionEvaluator(labelCol='duration')
pipeline = Pipeline(stages=[indexer, onehot, assembler, regression])
default_model = pipeline.fit(flights_train)
predictions = default_model.transform(flights_test)

print('The default model has RMSE of %f on testing data.' % evaluator.evaluate(predictions))

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel

params = ParamGridBuilder()\
             .addGrid(regression.regParam, [0.01, 0.1, 1.0, 10.0])\
             .addGrid(regression.elasticNetParam, [0.0, 0.5, 1.0])\
             .build()
			 
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=params, evaluator=evaluator, numFolds=5)
cv = cv.fit(flights_train)

The default model has RMSE of 10.793013 on testing data.


In [ ]:
# import os, pickle

# #cv.bestModel.write().overwrite().save('./flights-pipeline-best-model')

# from os import listdir, remove
# from shutil import rmtree

# # try:
# #      remove('flights-pipeline-best-model.zip')
# # except FileNotFoundError:
# #      pass
	 
# # os.system('zip -r ./flights-pipeline-best-model.zip ./flights-pipeline-best-model')
# # rmtree('./flights-pipeline-best-model')
# with open('flights-pipeline-average-metrics.pickle', 'wb') as f:
#      pickle.dump(cv.avgMetrics, f)
	 
# os.system('unzip flights-pipeline-best-model.zip')
# pipeline = PipelineModel.load('./flights-pipeline-best-model')
# cv = CrossValidatorModel(pipeline)
# cv.avgMetrics = pickle.load(open('flights-pipeline-average-metrics.pickle', 'rb'))

In [ ]:
# Get the best model from cross validation
best_model = cv.____

# Look at the stages in the best model
print(best_model.____)

# Get the parameters for the LinearRegression object in the best model
best_model.____.extractParamMap()

# Generate predictions on testing data using the best model then calculate RMSE
predictions = ____.____(____)
print("RMSE =", ____.____(____))